In [ ]:
import pandas as pd

In [ ]:
df = pd.read_excel('/Users/ngocnguyen/Downloads/cleaned_2018.xlsx')

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Filter for January to March (Q1) of 2018
may = df[(df['date'].dt.year == 2018) & (df['date'].dt.month.isin([5]))]

# Save the filtered dataset
#may.to_csv("may2018.csv", index=False)

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Load model and tokenizer
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# ✅ Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def analyze_sentiment(text):
    # Tokenize and prepare the input, move to GPU
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(device)

    # Perform inference on GPU
    with torch.no_grad():
        outputs = model(**inputs)

    # Get the predicted sentiment
    sentiment_scores = outputs.logits
    predicted_class = torch.argmax(sentiment_scores, dim=1).item()

    # Convert zero-indexed class to a 1-5 sentiment score
    sentiment_value = predicted_class + 1

    return sentiment_value


In [ ]:
may["sentiment_bert"] = may["article_title"].apply(analyze_sentiment)

In [ ]:
may.info()

In [ ]:
may.head(20)

In [ ]:
grouped_may = may.groupby(['date', 'stock_symbol'])['sentiment_bert'].mean().reset_index()
grouped_may.head()

In [ ]:
# Group by 'date' and 'stock_symbol', then pivot to have stock symbols as columns
grouped_may = may.groupby(['date', 'stock_symbol'])['sentiment_bert'].mean().unstack()

# Reset index to keep 'date' as a column
grouped_may.reset_index(inplace=True)

grouped_may.head(15)

In [ ]:
grouped_may.to_excel("/Users/ngocnguyen/Desktop/CPP/Capstone/grouped_df.xlsx", index=False)